# Tracing and the Dev Viewer

A multi-step agent's final output is often just a paragraph of text — but the interesting things happened along the way: which classification the model chose, which route it picked, how long each LLM call took, what the raw JSON was. NOOA has automatic tracing built in: every method invocation, whether Python or LLM-backed, becomes a span in a nested tree.

In this tutorial we'll build a small `SupportTicketAgent` that classifies, routes, and drafts a reply. Then we'll turn on JSONL tracing, read the spans back with plain Python, opt a helper out with `@no_trace`, and point at the live dev viewer for interactive exploration.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. Replace `"your-api-key"` with a real key for hosted providers; local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/openai/openai/gpt-5.5", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## A Multi-Step Support Ticket Agent

Our agent takes a raw support ticket and produces a drafted reply. Internally it does three things: classify the ticket, decide where to route it, then draft the response. Each step is a `PredictStrategy` method with a typed Pydantic output, so we can chain them cleanly.

The public entry point is `handle(ticket)` — a plain async method that orchestrates the three LLM calls.

In [ ]:
from typing import Literal
from pydantic import BaseModel
from nooa import Agent, strategy
from nooa.strategies import PredictStrategy


class Classification(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    severity: Literal["low", "medium", "high"]


class Route(BaseModel):
    team: str
    priority: Literal["p0", "p1", "p2", "p3"]
    sla_hours: int


class DraftReply(BaseModel):
    subject: str
    body: str

In [ ]:
class SupportTicketAgent(Agent, llm=model):
    """You are a support triage assistant. You classify incoming tickets, decide where they should go, and draft an initial reply to the customer."""

    @strategy(PredictStrategy())
    async def classify(self, ticket: str) -> Classification:
        """Classify the ticket into a category and severity."""
        ...

    @strategy(PredictStrategy())
    async def route(self, ticket: str, cls: Classification) -> Route:
        """Choose the team, priority, and SLA (in hours) for the ticket based on its classification."""
        ...

    @strategy(PredictStrategy())
    async def draft_reply(self, ticket: str, cls: Classification, route: Route) -> DraftReply:
        """Draft a short, professional first-response email to the customer. Acknowledge the issue and set expectations based on the SLA."""
        ...

    async def handle(self, ticket: str) -> DraftReply:
        cls = await self.classify(ticket)
        route = await self.route(ticket, cls)
        return await self.draft_reply(ticket, cls, route)

In [ ]:
tickets = [
    "My last invoice charged me twice for the Pro plan. Can you refund the duplicate?",
    "The dashboard has been showing a 500 error on the Analytics tab since yesterday morning.",
    "I'd like to change the email address on my account but the settings page won't save.",
]

agent = SupportTicketAgent()
for ticket in tickets:
    reply = await agent.handle(ticket)
    print("---")
    print(f"Ticket: {ticket}")
    print(f"Reply subject: {reply.subject}")
    print(f"Reply body:    {reply.body}")

The output is a polished reply — and that's precisely the problem when something goes wrong. You cannot see which category the classifier chose, which team the router picked, or how long each step took. From the outside, three separate LLM calls collapsed into a single string.

> Takeaway: as soon as an agent has more than one internal step, the final output stops being enough to debug it. Tracing gives you the intermediate reasoning back.

## Turning On JSONL Tracing

The simplest exporter writes every span to a JSONL file. One line per span, standard OTLP JSON format — you can read it with the `json` module and no extra dependencies.

Call `enable_tracing` once, before your agent runs. From that point on, every method call on every `Agent` subclass is captured.

In [ ]:
import tempfile
from pathlib import Path
from nooa.tracing import enable_tracing, exporters, flush_traces

trace_dir = Path(tempfile.mkdtemp(prefix="nooa_traces_"))
enable_tracing(exporters=[exporters.jsonl(trace_dir)])
print(f"Traces will be written to: {trace_dir}")

In [ ]:
agent = SupportTicketAgent()
reply = await agent.handle(
    "My last invoice charged me twice for the Pro plan. Can you refund the duplicate?"
)
print(f"Reply: {reply.subject}")

# Flush pending spans to disk before we read the file back.
flush_traces()

Let's list what got written, and then pretty-print one span so you can see the shape.

In [ ]:
import json

trace_files = sorted(trace_dir.glob("*.jsonl"))
print(f"{len(trace_files)} trace file(s) written:")
for f in trace_files:
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

spans = [json.loads(line) for line in trace_files[0].read_text().splitlines() if line.strip()]
print(f"\n{len(spans)} spans in {trace_files[0].name}")

In [ ]:
# Grab the first span and show a slimmed-down view of the interesting fields.
span = spans[0]
resource_spans = span["resourceSpans"][0]["scopeSpans"][0]["spans"][0]
print("name:            ", resource_spans["name"])
print("span_id:         ", resource_spans["spanId"])
print("parent_span_id:  ", resource_spans.get("parentSpanId", "(root)"))
print("attributes:")
for attr in resource_spans.get("attributes", [])[:6]:
    print(f"  {attr['key']}: {list(attr['value'].values())[0]}")

Each span carries a name (the method), an ID, an optional parent ID, timings, and attributes. Because `handle` calls `classify`, `route`, and `draft_reply`, the parent/child links let you rebuild the call tree offline. This is enough to power dashboards, replay tools, or a diff between two agent versions.

## Opting Out with `@no_trace`

By default, NOOA traces every method on an `Agent` subclass — public, private (`_helper`), and dunder alike. That's usually what you want, but not always. A rate-limit check that fires every request, or a `__repr__` used in logging, will flood your traces with uninteresting spans.

Apply `@no_trace` to opt a single method out. Below, `_check_rate_limit` is silent, while `_sanitise` — also private — is still traced because we haven't opted it out.

In [ ]:
from nooa import no_trace


class QuietTicketAgent(Agent, llm=model):
    """You are a support triage assistant."""

    @no_trace
    def _check_rate_limit(self) -> bool:
        """A noisy helper that runs on every request. Not worth a span."""
        return True

    def _sanitise(self, ticket: str) -> str:
        """Strip whitespace and cap length. Traced by default."""
        return ticket.strip()[:2000]

    @strategy(PredictStrategy())
    async def classify(self, ticket: str) -> Classification:
        """Classify the ticket into a category and severity."""
        ...

    async def handle(self, ticket: str) -> Classification:
        self._check_rate_limit()
        clean = self._sanitise(ticket)
        return await self.classify(clean)

In [ ]:
# Fresh trace dir so we can count spans cleanly.
trace_dir2 = Path(tempfile.mkdtemp(prefix="nooa_traces_optout_"))
enable_tracing(exporters=[exporters.jsonl(trace_dir2)])

quiet = QuietTicketAgent()
await quiet.handle("  Please cancel my subscription.  ")
flush_traces()

span_names = []
for f in trace_dir2.glob("*.jsonl"):
    for line in f.read_text().splitlines():
        if not line.strip():
            continue
        payload = json.loads(line)
        for rs in payload.get("resourceSpans", []):
            for ss in rs.get("scopeSpans", []):
                for sp in ss.get("spans", []):
                    span_names.append(sp["name"])

print("Traced method spans:")
for name in span_names:
    print(f"  - {name}")

You should see spans for `handle`, `_sanitise`, and `classify`, but no span for `_check_rate_limit`. The decorator is your one-line escape hatch: keep the default "trace everything" behavior, and mute what you don't want.

## The Dev Viewer

JSONL is great for automation, but for interactive exploration NOOA ships a local dev viewer: a small server that receives spans over OTLP and renders them as a live tree with timings, inputs, and outputs. It runs on port **5001**.

Start it in a separate terminal (not inside the notebook — Jupyter isn't a great host for a long-running HTTP server):

```bash
nooa start-dev
```

Then, in your Python session, point the tracer at it with the `local_otlp` exporter:

```python
from nooa.tracing import enable_tracing, exporters

enable_tracing(exporters=[exporters.local_otlp()])
```

Open `http://localhost:5001` in your browser and rerun the agent. Every method call, LLM invocation, and tool step streams into the viewer as it happens.

### Combining Exporters

`enable_tracing` accepts a list, so you can send the same spans to multiple sinks in one shot — for example, the live viewer during development plus a JSONL archive on disk:

```python
enable_tracing(exporters=[
    exporters.local_otlp(),
    exporters.jsonl("./traces"),
])
```

Other exporters are available for external OTLP collectors (`exporters.otlp(endpoint)`), Langfuse (`exporters.langfuse()`), and the terminal (`exporters.console()`). Pick whichever combination matches your workflow.

> Takeaway: tracing in NOOA is a single call. Default exporters cover the common cases (files, local viewer, console, external OTLP), and `@no_trace` is your fine-grained opt-out. You should almost never need to instrument anything by hand.

## Recap

- `enable_tracing(exporters=[...])` turns on automatic capture of every method span.
- `exporters.jsonl(dir)` writes standard OTLP JSON lines you can read with the `json` module.
- `exporters.local_otlp()` streams to the `nooa start-dev` viewer on port 5001.
- Multiple exporters can be combined in a single `enable_tracing` call.
- `@no_trace` opts a specific method out — private and dunder methods are traced by default.